In [1]:
import os
import json
from utils import set_seed, generate_namespace
from transformers import Trainer, TrainingArguments

from lora_fine_tune.model_local import LocalLM
from lora_fine_tune.data_utils import build_dataset

%matplotlib inline
%load_ext autoreload
%autoreload 2

In [2]:
cfg = generate_namespace(path=f"../config.yaml")
print(json.dumps(vars(cfg), indent=2))

set_seed(cfg.seed)

{
  "model_name": "gpt2",
  "api_model_name": "gpt-4.1-nano",
  "seed": 42,
  "max_token_length": 128,
  "lr": 0.0001,
  "warmup_ratio": 0.0,
  "epochs": 500,
  "train_batch_size": 1,
  "strategy": "steps",
  "logging_steps": 5,
  "res_path": "../results/"
}


In [3]:
model = LocalLM(cfg.model_name)

total_params = sum(p.numel() for p in model.model.parameters())
trainable_params = sum(p.numel() for p in model.model.parameters() if p.requires_grad)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

Total parameters: 124,439,808
Trainable parameters: 124,439,808


/opt/miniconda3/envs/lora-fine-tune-env/lib/python3.12/site-packages/peft/tuners/lora/layer.py:2285: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


In [4]:
model.lora_model.print_trainable_parameters()

trainable params: 294,912 || all params: 124,734,720 || trainable%: 0.2364


In [5]:
dataset = build_dataset()
data_tokenized = model.tokenize_dataset(dataset, cfg.max_token_length, padding=False)
data_collator = model.data_collator()
print(data_tokenized)

Map:   0%|          | 0/6 [00:00<?, ? examples/s]

Dataset({
    features: ['text', 'input_ids', 'attention_mask'],
    num_rows: 6
})


In [6]:
training_args = TrainingArguments(
    output_dir=cfg.res_path,
    overwrite_output_dir=True,
    report_to="none",
    save_strategy="no",
    logging_strategy= "steps",
    logging_steps=cfg.logging_steps,
    learning_rate=cfg.lr,
    per_device_train_batch_size=cfg.train_batch_size,
    num_train_epochs=cfg.epochs,
    warmup_ratio=cfg.warmup_ratio,
    seed=cfg.seed
)

trainer = Trainer(
    model=model.lora_model,
    args=training_args,
    train_dataset=data_tokenized,
    data_collator=data_collator
)

trainer.train()

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
5,3.877000
10,3.217300
15,3.870400
20,3.074100
25,4.290900
30,3.394500
35,3.251900
40,3.662500
45,3.197400
50,3.364600


TrainOutput(global_step=3000, training_loss=0.400437540302674, metrics={'train_runtime': 212.1627, 'train_samples_per_second': 14.14, 'train_steps_per_second': 14.14, 'total_flos': 33286855680000.0, 'train_loss': 0.400437540302674, 'epoch': 500.0})

In [8]:
eval_prompts = [
    "Q: How do transformers work?\nA:",
    "Q: What does physics explain?\nA:",
    "Q: What do machine learning models do?\nA:",
    "Q: What is a language model?\nA:",
    "Q: What is overfitting?\nA:",
    "Q: What is a LoRA adapter?\nA:",
]

for e in eval_prompts:
    print(f"Model output:\n{model.generate(e)}\n")
    print(f"LoRA model output:\n{model.generate(e, use_lora=True)}\n\n")

Model output:
Q: How do transformers work?
A: A transformer is a set of components. The first component on the input (usually an individual resistor)

LoRA model output:
Q: How do transformers work?
A: Transformers process sequences using self-attention mechanisms. Self attention allows tokens in a sequence to attend each


Model output:
Q: What does physics explain?
A: Physics is a theory of the universe. It involves particles with different properties that move through space and time

LoRA model output:
Q: What does physics explain?
A: Physics explains natural phenomena using mathematical models. Model definitions allow intelligent agents to observe patterns in sequence. Einstein


Model output:
Q: What do machine learning models do?
A: It is a lot of things. We often see many different aspects on the side that we can use

LoRA model output:
Q: What do machine learning models do?
A: Machine-learning model designs learn patterns from data. Model layers process sequences using self-att